# Bronze | Qualidade de dados
Valida o **último lote** de cada tabela Bronze (a Bronze é append-only; lotes antigos não são reavaliados).

| Regra | Severidade |
|---|---|
| `LOTE_VAZIO` (último lote sem registros) | BLOQUEANTE |
| `METADADOS_NULOS` (`lote_id`, `data_ingestao`, `fonte`) | BLOQUEANTE |
| `CHAVE_NULA` | BLOQUEANTE |
| `CHAVE_DUPLICADA` no lote | INFORMATIVA (a Silver deduplica) |

As regras são persistidas antes do gate; a falha é registrada em `pipeline_runs` e propagada com `raise`.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
REGRAS = [
    ("uf_raw", ["ano", "sigla_uf", "serie", "rede"]),
    ("meta_alfabetizacao_brasil_raw", ["ano", "rede"]),
    ("meta_alfabetizacao_uf_raw", ["ano", "sigla_uf", "rede"]),
    ("meta_alfabetizacao_municipio_raw", ["ano", "id_municipio"]),
    ("municipio_raw", ["ano", "id_municipio", "serie", "rede"]),
    ("alunos_raw", ["ano", "id_municipio", "id_escola", "id_aluno"]),
    ("dicionario_raw", ["id_tabela", "nome_coluna", "chave"]),
    ("diretorio_municipio_raw", ["id_municipio"]),
    ("diretorio_uf_raw", ["id_uf"]),
]

_m = iniciar_etapa()
dq = RegistroDQ("BRONZE")

try:
    for tabela, chaves in REGRAS:
        full = f"workspace.bronze.{tabela}"
        if not spark.catalog.tableExists(full):
            dq.registrar(full, "TABELA_AUSENTE", "BLOQUEANTE", 1, "Tabela Bronze não existe")
            continue

        ultimo = spark.sql(f"""
            SELECT lote_id FROM {full}
            WHERE lote_id IS NOT NULL AND data_ingestao IS NOT NULL
            ORDER BY data_ingestao DESC LIMIT 1""").first()
        if ultimo is None:
            dq.registrar(full, "LOTE_VAZIO", "BLOQUEANTE", 1, "Nenhum lote válido encontrado")
            continue
        lote = ultimo["lote_id"]
        det = f"lote_id={lote}; chave={','.join(chaves)}"

        total = spark.sql(f"SELECT COUNT(*) n FROM {full} WHERE lote_id = '{lote}'").first()["n"]
        dq.registrar(full, "LOTE_VAZIO", "BLOQUEANTE", 1 if total == 0 else 0, f"{det}; registros_no_lote={total}")

        meta_nulos = spark.sql(f"""SELECT COUNT(*) n FROM {full} WHERE lote_id = '{lote}'
                                   AND (data_ingestao IS NULL OR fonte IS NULL)""").first()["n"]
        dq.registrar(full, "METADADOS_NULOS", "BLOQUEANTE", meta_nulos, det)

        cond = " OR ".join(f"`{k}` IS NULL" for k in chaves)
        nulas = spark.sql(f"SELECT COUNT(*) n FROM {full} WHERE lote_id = '{lote}' AND ({cond})").first()["n"]
        dq.registrar(full, "CHAVE_NULA", "BLOQUEANTE", nulas, det)

        lista = ", ".join(f"`{k}`" for k in chaves)
        dups = spark.sql(f"""SELECT COUNT(*) n FROM (SELECT {lista} FROM {full} WHERE lote_id = '{lote}'
                             GROUP BY {lista} HAVING COUNT(*) > 1)""").first()["n"]
        dq.registrar(full, "CHAVE_DUPLICADA", "INFORMATIVA", dups, det)

    dq.gate()
except Exception as exc:
    registrar_execucao("BRONZE", "dq_bronze", _m, status="FALHA", erro=exc)
    raise
registrar_execucao("BRONZE", "dq_bronze", _m, status="SUCESSO")